In [0]:
# =============================================================================
# NB00b — Patch ionic_radius_shannon_vi in ml_perovskites.atom_props_curated
# =============================================================================
# Objective:
#   Fill the ionic_radius_shannon_vi column (currently all NULL) with
#   Shannon (1976) effective ionic radii in Angstrom, coordination number VI
#   (octahedral), for the most common oxidation states of each element.
#
# Source:
#   Shannon, R.D. (1976). Acta Crystallographica A, 32, 751-767.
#   DOI: 10.1107/S0567739476001551
#   Values for CN=VI, most common oxidation state for perovskite cations.
#   O²⁻ radius used for Goldschmidt tolerance factor calculation.
#
# Run ONCE after NB00, before NB02b.
# =============================================================================

import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

SCHEMA = "ml_perovskites"
TABLE  = "atom_props_curated"

spark = SparkSession.builder.getOrCreate()
spark.conf.set("spark.sql.shuffle.partitions", "8")

# ---------------------------------------------------------------------------
# Shannon effective ionic radii [Angstrom], CN=VI, typical oxidation state
# Reference: Shannon (1976), Acta Cryst. A32, 751-767
# Coverage: all A-site and B-site elements found in ABO3 perovskites
# ---------------------------------------------------------------------------
SHANNON_VI = {
    # Anions
    "O":  1.40,   # O2-, CN=VI
    "N":  1.46,   # N3-, CN=VI
    "F":  1.33,   # F-, CN=VI
    "Cl": 1.81,   # Cl-, CN=VI
    "Br": 1.96,   # Br-, CN=VI
    "I":  2.20,   # I-, CN=VI
    "S":  1.84,   # S2-, CN=VI
    # A-site cations (typically 12-coord in perovskite, but VI used here)
    "Li": 0.76,   # Li+, CN=VI
    "Na": 1.02,   # Na+, CN=VI
    "K":  1.38,   # K+, CN=VI
    "Rb": 1.52,   # Rb+, CN=VI
    "Cs": 1.67,   # Cs+, CN=VI
    "Be": 0.45,   # Be2+, CN=VI
    "Mg": 0.72,   # Mg2+, CN=VI
    "Ca": 1.00,   # Ca2+, CN=VI
    "Sr": 1.18,   # Sr2+, CN=VI
    "Ba": 1.35,   # Ba2+, CN=VI
    "Ra": 1.48,   # Ra2+, CN=VI
    "Sc": 0.745,  # Sc3+, CN=VI
    "Y":  0.90,   # Y3+, CN=VI
    "La": 1.032,  # La3+, CN=VI
    "Ce": 1.01,   # Ce3+, CN=VI (Ce4+=0.87)
    "Pr": 0.99,   # Pr3+, CN=VI
    "Nd": 0.983,  # Nd3+, CN=VI
    "Pm": 0.97,   # Pm3+, CN=VI
    "Sm": 0.958,  # Sm3+, CN=VI
    "Eu": 0.947,  # Eu3+, CN=VI (Eu2+=1.17)
    "Gd": 0.938,  # Gd3+, CN=VI
    "Tb": 0.923,  # Tb3+, CN=VI (Tb4+=0.76)
    "Dy": 0.912,  # Dy3+, CN=VI
    "Ho": 0.901,  # Ho3+, CN=VI
    "Er": 0.890,  # Er3+, CN=VI
    "Tm": 0.880,  # Tm3+, CN=VI
    "Yb": 0.868,  # Yb3+, CN=VI
    "Lu": 0.861,  # Lu3+, CN=VI
    "Ac": 1.12,   # Ac3+, CN=VI
    "Th": 0.94,   # Th4+, CN=VI
    "Pb": 1.19,   # Pb2+, CN=VI
    "Bi": 1.03,   # Bi3+, CN=VI
    "Tl": 1.50,   # Tl+, CN=VI
    "In": 0.80,   # In3+, CN=VI
    # B-site cations (transition metals and post-transition)
    "Ti": 0.605,  # Ti4+, CN=VI
    "Zr": 0.72,   # Zr4+, CN=VI
    "Hf": 0.71,   # Hf4+, CN=VI
    "V":  0.54,   # V5+, CN=VI (V4+=0.58)
    "Nb": 0.64,   # Nb5+, CN=VI
    "Ta": 0.64,   # Ta5+, CN=VI
    "Cr": 0.615,  # Cr3+, CN=VI
    "Mo": 0.59,   # Mo6+, CN=VI (Mo4+=0.65)
    "W":  0.60,   # W6+, CN=VI
    "Mn": 0.645,  # Mn3+, CN=VI (Mn2+=0.83, Mn4+=0.53)
    "Fe": 0.645,  # Fe3+, CN=VI (Fe2+=0.78)
    "Co": 0.545,  # Co3+, CN=VI (Co2+=0.745)
    "Ni": 0.69,   # Ni2+, CN=VI
    "Cu": 0.73,   # Cu2+, CN=VI
    "Zn": 0.74,   # Zn2+, CN=VI
    "Ga": 0.62,   # Ga3+, CN=VI
    "Ge": 0.53,   # Ge4+, CN=VI
    "Sn": 0.69,   # Sn4+, CN=VI
    "Al": 0.535,  # Al3+, CN=VI
    "Si": 0.40,   # Si4+, CN=VI
    "Ru": 0.62,   # Ru4+, CN=VI
    "Rh": 0.665,  # Rh3+, CN=VI
    "Pd": 0.86,   # Pd2+, CN=VI
    "Os": 0.63,   # Os4+, CN=VI
    "Ir": 0.625,  # Ir4+, CN=VI
    "Pt": 0.625,  # Pt4+, CN=VI
    "Au": 0.85,   # Au3+, CN=VI
    "Ag": 1.15,   # Ag+, CN=VI
    "Cd": 0.95,   # Cd2+, CN=VI
    "Hg": 1.02,   # Hg2+, CN=VI
    "Re": 0.63,   # Re4+, CN=VI
    "U":  0.89,   # U4+, CN=VI
    "Np": 0.87,   # Np4+, CN=VI
    "Pu": 0.86,   # Pu4+, CN=VI
}

# ---------------------------------------------------------------------------
# 1. Load current table
# ---------------------------------------------------------------------------
ap = spark.table(f"{SCHEMA}.{TABLE}").toPandas()
print(f"Current shape: {ap.shape}")
print(f"ionic_radius_shannon_vi non-null before patch: {ap['ionic_radius_shannon_vi'].notna().sum()}")

# ---------------------------------------------------------------------------
# 2. Patch values
# ---------------------------------------------------------------------------
ap["ionic_radius_shannon_vi"] = ap["Element"].map(SHANNON_VI)

covered = ap["ionic_radius_shannon_vi"].notna().sum()
missing = ap[ap["ionic_radius_shannon_vi"].isna()]["Element"].tolist()
print(f"\nionic_radius_shannon_vi non-null after patch: {covered} / {len(ap)}")
if missing:
    print(f"Elements without Shannon radius (rare gases + actinides without data): {missing}")

# ---------------------------------------------------------------------------
# 3. Overwrite Delta table
# ---------------------------------------------------------------------------
for col in ap.columns:
    if col != "Element":
        ap[col] = pd.to_numeric(ap[col], errors="coerce").astype(float)

ap_clean = ap.replace({np.nan: None})
spark_df = spark.createDataFrame(ap_clean)

(spark_df.write
 .format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable(f"{SCHEMA}.{TABLE}"))

# ---------------------------------------------------------------------------
# 4. Verify
# ---------------------------------------------------------------------------
check = spark.table(f"{SCHEMA}.{TABLE}")
rc = check.count()
non_null = check.filter(F.col("ionic_radius_shannon_vi").isNotNull()).count()
print(f"\n✓ Written: {SCHEMA}.{TABLE} ({rc} rows)")
print(f"  ionic_radius_shannon_vi populated: {non_null} / {rc} elements")
print("\n=== NB00b COMPLETE ===")
print("Now re-run NB02b to regenerate X_material with structural descriptors.")